실행 방법 요약

  1. 가상환경 활성화 및 Jupyter 실행

  conda activate medgemma
  jupyter notebook

  2. Hugging Face 토큰 설정

  노트북을 열기 전에 터미널에서:
  export HF_TOKEN="your_huggingface_token_here"

  또는 노트북 내에서 두 번째 코드 셀에 직접 토큰을 입력할 수도 있습니다.

  3. 노트북 실행 순서

  1. radiology-report-3_ko_local.ipynb 열기
  2. 셀을 위에서부터 순서대로 실행

  노트북에 13개의 셀이 포함되어 있습니다:
  - 환경 설정 및 Hugging Face 로그인
  - MedGemma 모델 로드
  - ReXGradient 데이터셋 로드
  - Impression 생성 함수
  - 테스트 및 평가
  - ROUGE/BERTScore 계산
  - 결과 저장 및 분석

In [ ]:
!huggingface-cli login

In [ ]:
# 최고/최저 성능 예시 분석

# 최저 ROUGE-L 예시
worst_idx = np.argmin(rougeL_scores)

print("🔴 최저 성능 예시")
print(f"ROUGE-L: {rougeL_scores[worst_idx]:.3f}")
print(f"\n📝 Findings:\n{val_reports[worst_idx]['findings'][:500]}...")
print(f"\n🤖 생성된 Impression:\n{predictions[worst_idx]}")
print(f"\n✅ 실제 Impression:\n{references[worst_idx]}")

# 최고 ROUGE-L 예시
best_idx = np.argmax(rougeL_scores)

print("\n🟢 최고 성능 예시")
print(f"ROUGE-L: {rougeL_scores[best_idx]:.3f}")
print(f"\n📝 Findings:\n{val_reports[best_idx]['findings'][:500]}...")
print(f"\n🤖 생성된 Impression:\n{predictions[best_idx]}")
print(f"\n✅ 실제 Impression:\n{references[best_idx]}")

In [ ]:
# 결과 저장

import pandas as pd

results_df = pd.DataFrame({
    "findings": [r["findings"][:500] + "..." for r in val_reports[:len(predictions)]],
    "true_impression": references,
    "generated_impression": predictions,
    "rouge1": rouge1_scores,
    "rouge2": rouge2_scores,
    "rougeL": rougeL_scores,
    "bertscore_f1": F1.tolist()
})

# 결과 저장
output_path = "/home/jkuhm/src/MoltsBot-Source/medgemma-clone/medgemma/challenge/evaluation_results.csv"
results_df.to_csv(output_path, index=False)
print(f"✅ 결과가 저장되었습니다: {output_path}")

# 상위 5개 예시 표시
print("\n📋 상위 5개 예시:")
print(results_df.head(5))

In [ ]:
# BERTScore 계산 (GPU 메모리 해제 후 실행)

from bert_score import score as bert_score

# GPU 메모리 해제
model.to("cpu")
torch.cuda.empty_cache()
print("🔄 GPU 메모리 해제 완료")

# BERTScore 계산
P, R, F1 = bert_score(predictions, references, lang="en", verbose=True)

print(f"\n📊 BERTScore F1 (평균): {F1.mean().item():.3f} (±{F1.std().item():.3f})")

In [ ]:
# ROUGE 점수 계산

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)
    rouge1_scores.append(scores['rouge1'].fmeasure)
    rouge2_scores.append(scores['rouge2'].fmeasure)
    rougeL_scores.append(scores['rougeL'].fmeasure)

print("📊 ROUGE 점수 (평균 ± 표준편차):")
print(f"ROUGE-1: {np.mean(rouge1_scores):.3f} (±{np.std(rouge1_scores):.3f})")
print(f"ROUGE-2: {np.mean(rouge2_scores):.3f} (±{np.std(rouge2_scores):.3f})")
print(f"ROUGE-L: {np.mean(rougeL_scores):.3f} (±{np.std(rougeL_scores):.3f})")

In [ ]:
# 검증 데이터셋에서 평가

import numpy as np
from tqdm import tqdm

# 검증 데이터에서 50개 샘플링
val_sample = val_data.shuffle(seed=42).select(range(min(50, len(val_data))))

val_reports = []
for item in val_sample:
    val_reports.append({
        "findings": item.get("Findings", ""),
        "impression": item.get("Impression", "")
    })

# 결과 저장
predictions = []
references = []

print("🔄 Impression 생성 중...")
for i, item in enumerate(tqdm(val_reports[:50], desc="Generating")):
    if item["findings"] and item["impression"]:
        pred = generate_impression(item["findings"])
        predictions.append(pred)
        references.append(item["impression"])

print(f"\n✅ {len(predictions)}개 Impression 생성 완료")

In [ ]:
# 테스트: 첫 번째 보고서에 대한 Impression 생성

test_findings = reports[0]["findings"]
generated = generate_impression(test_findings)
true_impression = reports[0]["impression"]

print("🤖 생성된 Impression:\n", generated)
print("\n✅ 실제 Impression:\n", true_impression)

In [ ]:
# Impression 생성 함수 정의

def generate_impression(findings_text):
    """Findings 섹션에서 Impression을 생성합니다."""
    messages = [
        {
            "role": "user",
            "content": f"""You are a board-certified radiologist.
Given the "Findings" section of a chest X-ray report, write a concise, professional "Impression" section.
The Impression should:
- List the most important diagnoses first
- Use standardized terminology
- Be suitable for a referring physician

Findings:
{findings_text}"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

In [ ]:
# 샘플 데이터 확인

# 빠른 테스트를 위해 처음 10개 예시 가져오기
small_sample = train_data.select(range(min(10, len(train_data))))

reports = []
for item in small_sample:
    findings = item.get("Findings", "")
    impression = item.get("Impression", "")
    reports.append({"findings": findings, "impression": impression})

print(f"📝 {len(reports)}개 보고서 로딩 완료")
print("\n📋 예시:")
print("Findings:", reports[0]["findings"][:500] + "...")
print("Impression:", reports[0]["impression"])

In [ ]:
# ReXGradient-160K 데이터셋 로드

from datasets import load_dataset

print("🔄 데이터셋 로딩 중...")
dataset = load_dataset(
    "rajpurkarlab/ReXGradient-160K",
    "metadata",  # 이미지 없이 텍스트만 로드
    token=hf_token if hf_token else True
)

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print(f"\n✅ 데이터셋 로딩 완료!")
print(f"📊 Train: {len(train_data):,} reports")
print(f"📊 Validation: {len(val_data):,} reports")
print(f"📊 Test: {len(test_data):,} reports")

In [ ]:
# MedGemma 모델 로드

model_id = "google/medgemma-4b-it"

print("🔄 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    token=hf_token if hf_token else True
)

print("🔄 모델 로딩 중 (1-2분 소요됩니다)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=hf_token if hf_token else True
)
print("✅ 모델 로딩 완료!")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login

# 환경 변수에서 토큰 읽기
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ Hugging Face에 로그인되었습니다.")
else:
    print("⚠️ HF_TOKEN 환경 변수가 설정되지 않았습니다.")
    print("터미널에서 실행: export HF_TOKEN='your_token_here'")
    print("또는: huggingface-cli login")

# CUDA 사용 가능 여부 확인
print(f"\n🔧 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔧 CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"🔧 CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# 필수 패키지 설치 (이미 설치되어 있으면 건너뛰세요)
# !pip install -q datasets huggingface_hub transformers accelerate rouge-score bert-score tqdm

# MedGemma 방사선 보고서 생성 (로컬 환경)

이 노트북은 MedGemma-4b-it 모델을 사용하여 방사선 보고서의 Findings 섹션에서 Impression 섹션을 생성합니다.

## 사전 요구사항

1. **Hugging Face 토큰**: 다음 모델과 데이터셋에 접근하려면 Hugging Face 계정에서 토큰이 필요합니다:
   - google/medgemma-4b-it (gated model)
   - rajpurkarlab/ReXGradient-160K (gated dataset)

2. **Hugging Face CLI 로그인**: 터미널에서 `huggingface-cli login`을 실행하세요.